# Environment setup

This notebook is run **once**, before anything else. It verifies that the local
machine can reach Databricks, that the Lakebase branch answers, and it creates the
Unity Catalog namespace the rest of the project writes into.

Nothing here is part of the pipeline. If it fails, nothing downstream will work,
and the error will be far less obvious when it shows up in notebook 01.

## How this project runs

The notebooks live on a local machine and execute against Databricks through
**Databricks Connect**: the code is written here, the computation happens there.
That keeps the editor, the version control and the debugger local, while the
data never leaves the workspace.

Three services are involved, and they are not interchangeable:

| Service | What it is | What it holds here |
|---|---|---|
| **Lakebase** | Postgres 17 inside Databricks | The operational tables: customers, predictions, model runs |
| **Unity Catalog** | Governance and namespace | The Delta tables, in bronze / silver / gold |
| **Serverless compute** | Spark | Everything the notebooks execute |

Lakebase is the transactional side: single rows, foreign keys, low latency.
Delta is the analytical side: columns, history, large scans. They are two engines
because they solve two different problems, and an analytical scan over the
transactional database would compete with the writes of the system that actually
depends on it.

## 1 · Connect

`bootstrap()` lives in `src/config.py` and does four things: loads the profile from
`.env`, opens the Spark session and the SDK client, pins the Lakebase branch **by
name**, and resolves the Postgres host from the endpoint.

Two details in there are worth knowing about, because both cause failures that
look like something else:

- The VS Code Databricks extension injects `DATABRICKS_HOST` and friends into the
  environment, and those **silently override** the profile. They are removed first.
- The branch is selected by name rather than by taking the first one the API
  returns. With more than one branch that order is not guaranteed, and picking the
  wrong one would mean writing to production while believing otherwise.

No credential is ever printed. A token pasted into a saved output and pushed to
GitHub is a real leak, not a hypothetical one.

In [3]:
import sys
sys.path.append("../src")

from config import bootstrap

ctx = bootstrap()
spark, w = ctx.spark, ctx.w

connected to Databricks
  branch   : sandbox
  catalog  : bank_churn_eng
  identity : juzoushio@...


## 2 · Verify the client versions

`w.postgres` is the Lakebase API. It only exists from a certain SDK version onward,
so an outdated `databricks-sdk` fails with an attribute error that says nothing
about the real cause.

`protobuf` is the other usual suspect. The SDK excludes several specific versions,
and outside that range it fails when serialising, with a message that never
mentions protobuf.

In [4]:
import importlib.metadata as meta

for package in ("databricks-sdk", "databricks-connect", "protobuf", "pg8000"):
    try:
        print(f"  {package:20} {meta.version(package)}")
    except meta.PackageNotFoundError:
        print(f"  {package:20} NOT INSTALLED")

print(f"\n  Lakebase API available: {hasattr(w, 'postgres')}")

  databricks-sdk       0.110.0
  databricks-connect   18.1.3
  protobuf             6.33.6
  pg8000               1.31.5

  Lakebase API available: True


## 3 · The Lakebase branch

A branch is a **copy-on-write fork** of the database. Creating one copies nothing:
it inherits schema and data through pointers to the same storage, and only writes
new blocks when something is modified. That is why it is instant and free until
you touch it.

The whole project runs on `sandbox`, never on `production`. If a rebuild goes
wrong, the branch is deleted and nothing happened.

In [5]:
from config import PROJECT_ID, BRANCH_NAME

branches = list(w.postgres.list_branches(f"projects/{PROJECT_ID}"))

print(f"branches in {PROJECT_ID}:")
for b in branches:
    short = b.name.split("/")[-1]
    marker = "  <-- in use" if short == BRANCH_NAME else ""
    print(f"  {short}{marker}")

print(f"\nendpoint : {ctx.endpoint.split('/')[-1]}")
print(f"host     : {ctx.pg_host.split('.')[0]}...")

branches in bank-churn-prediction:
  sandbox  <-- in use
  production

endpoint : primary
host     : ep-super-lake-d8p42pyq...


## 4 · Does Postgres answer?

The credential is an OAuth token valid for sixty minutes, generated on every
connection. Generating a fresh one each time is cheaper than reasoning about
whether the one from twenty minutes ago has expired.

`ssl_context` is the pg8000 equivalent of `sslmode=require`. Lakebase does not
accept unencrypted connections, so without it the handshake simply fails.

In [6]:
from config import PG_SCHEMA

print(ctx.query("SELECT version() AS v").iloc[0, 0][:60], "\n")

objetos = ctx.query("""
    SELECT table_name, table_type
    FROM   information_schema.tables
    WHERE  table_schema = %s
    ORDER  BY table_type, table_name
""", (PG_SCHEMA,))

if objetos.empty:
    print(f"schema '{PG_SCHEMA}' is empty -- run sql/01_landing.sql first")
else:
    print(f"objects in schema '{PG_SCHEMA}':")
    for _, r in objetos.iterrows():
        print(f"  {r.table_type:<12} {r.table_name}")

PostgreSQL 17.10 (29ad1b7) on x86_64-pc-linux-gnu, compiled  

objects in schema 'bank_churn':
  BASE TABLE   customer_predictions
  BASE TABLE   customers
  BASE TABLE   customers_raw
  BASE TABLE   geographies
  BASE TABLE   model_runs
  VIEW         v_latest_predictions


## 5 · The Unity Catalog namespace

Three schemas, and the split is not cosmetic. Each one answers a different question
about the data it holds.

| Schema | What lives there | The rule |
|---|---|---|
| `bronze` | The source, unchanged | Nothing is corrected here. Fidelity beats cleanliness |
| `silver` | Typed, clean, derived features | Written **after** the test set has been held out |
| `gold` | Predictions and tables the app reads | One writer per table, no exceptions |

The `bronze` / `silver` boundary is where the anti-leakage control sits. Feature
engineering happens in silver, after the split, so the test set cannot influence
the variables that will later be used to evaluate against it.

In [7]:
from config import UC_CATALOG

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {UC_CATALOG}
    COMMENT 'Bank customer churn prediction - local execution via Databricks Connect'
""")

SCHEMAS = {
    "bronze": "Immutable copy of the source. No transformations.",
    "silver": "Typed and clean, with deterministic derived features. Source for modelling.",
    "gold":   "Predictions, metrics and tables the application consumes.",
}

for name, comment in SCHEMAS.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{name} COMMENT '{comment}'")
    print(f"  {UC_CATALOG}.{name}")

  bank_churn_eng.bronze
  bank_churn_eng.silver
  bank_churn_eng.gold


## 6 · The volume that holds the source file

The CSV is not downloaded at run time. It sits in a Unity Catalog volume, which
makes the notebook reproducible without Kaggle credentials and puts the file under
the same governance as the tables.

The path is `/Volumes/<catalog>/<schema>/<volume>/<file>` and it only exists on
Databricks compute. From a local Python process `os.path.exists` on that path
returns `False` — the filesystem is remote, so it has to be reached through Spark
or through the Files API.

In [8]:
from config import VOLUME_PATH

spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.bronze.raw_data")

files = spark.sql(f"LIST '/Volumes/{UC_CATALOG}/bronze/raw_data/'").toPandas()

if files.empty:
    print("the volume is empty. Upload churn.csv through Catalog Explorer:")
    print(f"  {UC_CATALOG} > bronze > raw_data")
else:
    for _, r in files.iterrows():
        print(f"  {r['name']:<24} {r['size']:>10,} bytes")

print(f"\nexpected by the pipeline: {VOLUME_PATH}")

  churn.csv                   684,858 bytes

expected by the pipeline: /Volumes/bank_churn_eng/bronze/raw_data/churn.csv


## 7 · Final state

Everything below should be green before moving to notebook 00. If the volume is
still empty, upload `churn.csv` through Catalog Explorer and re-run section 6 —
that is the only step that cannot be done from code, because the file comes from
outside the workspace.

In [9]:
from config import cost_summary

checks = {
    "Spark session":     spark is not None,
    "Lakebase reachable": not ctx.query("SELECT 1 AS ok").empty,
    "Catalog exists":     not spark.sql(f"SHOW SCHEMAS IN {UC_CATALOG}").toPandas().empty,
    "Source file present": not files.empty,
}

for label, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'} {label}")

print(f"\neconomics fixed in notebook 00:\n  {cost_summary()}")
print("\nnext: 00_problem_definition.ipynb" if all(checks.values())
      else "\nresolve the items above before continuing")

  OK  Spark session
  OK  Lakebase reachable
  OK  Catalog exists
  OK  Source file present

economics fixed in notebook 00:
  value(TP)=145 EUR | cost(FP)=35 EUR | break-even p*=0.1944 | capacity=800

next: 00_problem_definition.ipynb
